1️⃣ Load Clean Data

In [3]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

df = pd.read_csv('../data/raw/PhiUSIIL_Phishing_URL_Dataset.csv')

drop_cols = ["FILENAME", "URL", "Domain", "Title"]
df = df.drop(columns=drop_cols)

2️⃣ Feature Grouping (CRITICAL STEP)

We split features into:

🟢 URL-safe (usable in MVP)

In [4]:
url_features = [
    "URLLength", "DomainLength", "IsDomainIP",
    "NoOfSubDomain", "NoOfLettersInURL", "LetterRatioInURL",
    "NoOfDegitsInURL", "DegitRatioInURL",
    "NoOfOtherSpecialCharsInURL", "SpacialCharRatioInURL",
    "HasObfuscation", "NoOfObfuscatedChar", "ObfuscationRatio",
    "IsHTTPS"
]

df_url = df[url_features + ["label"]]

df_url.to_csv('../data/processed/url_only_dataset.csv', index=False)

print("✅ URL-only dataset saved")

✅ URL-only dataset saved


🔴 Content-based (NOT usable in MVP API yet)

In [5]:
content_features = [
    "LineOfCode", "NoOfPopup", "NoOfiFrame", "HasFavicon",
    "HasExternalFormSubmit", "HasPasswordField",
    "NoOfJS", "NoOfImage", "NoOfCSS"
]

df_content = df[content_features + ["label"]]

df_content.to_csv('../data/processed/content_only_dataset.csv', index=False)

print("✅ Content-only dataset saved")

✅ Content-only dataset saved


In [6]:
df_full = df.copy()

df_full.to_csv('../data/processed/full_features_dataset.csv', index=False)

print("✅ Full feature dataset saved")

✅ Full feature dataset saved


🟡 Hybrid / suspicious signals

In [7]:
hybrid_features = [
    "URLSimilarityIndex", "DomainTitleMatchScore",
    "URLTitleMatchScore", "TLDLegitimateProb"
]

df_url_hybrid = df[url_features + hybrid_features + ["label"]]

df_url_hybrid.to_csv('../data/processed/url_hybrid_dataset.csv', index=False)

print("✅ URL + Hybrid dataset saved")

✅ URL + Hybrid dataset saved


3️⃣ Experiment 1 — URL Features Only (MVP Model)

In [8]:
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import classification_report

X = df[url_features]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = XGBClassifier()
model.fit(X_train, y_train)

preds = model.predict(X_test)

print("URL-only model")
print(classification_report(y_test, preds))

URL-only model
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     20103
           1       1.00      1.00      1.00     27056

    accuracy                           1.00     47159
   macro avg       1.00      1.00      1.00     47159
weighted avg       1.00      1.00      1.00     47159



4️⃣ Experiment 2 — URL + Hybrid

In [9]:
X = df[url_features + hybrid_features]

5️⃣ Experiment 3 — FULL FEATURES

In [10]:
X = df.drop("label", axis=1)